# 01 · Mosaico anual (42 bandas)

**O que esta etapa faz:** monta, para um ano e uma região, o mosaico anual que alimenta a classificação: mediana das bandas Landsat e de ~25 índices (vegetação, água, área construída, mistura espectral) mais percentis do EVI e do EBBI.

**Equivalente na Col 11 (versão de produção):** `collection 11/codes_producao/02_classificacao/users/edimilsonrodriguessantos/mapbiomas/Col10/classificacao/mosaic_production.js` (+ `index_lib.js`, `preProcessing_lib.js`, `renameBands.js` na mesma pasta).

**Fonte do mosaico** (em `config/col12.yaml`, campo `mosaic.source`):
- `on_the_fly`: calculado do Landsat na hora, como na Col 11;
- `asset`: lido de mosaicos pré-processados e salvos como asset.

Rode os blocos em ordem. Nada é exportado sem você rodar explicitamente o bloco 5.

## 0 · Preparação
No Colab, este bloco baixa o código do GitHub. No computador local, só aponta para a pasta do pacote.

In [ ]:
import sys, os
EM_COLAB = 'google.colab' in sys.modules
if EM_COLAB:
    !git clone -q -b col12 https://github.com/jpedrassoli/brazil-urban-area.git repo 2>/dev/null || (cd repo && git pull -q)
    %pip install -q -e "repo/lulc_30m_landsat/collection 12" geemap
    RAIZ = 'repo/lulc_30m_landsat/collection 12'
else:
    RAIZ = os.path.abspath('..')
sys.path.insert(0, RAIZ)

import ee
from mb_urban import auth, config, grids, mosaic, tasks
cfg = config.load()
print('pacote pronto em', RAIZ)

## 1 · Conta do Earth Engine
- Local: `conta1` (Gmail) ou `conta2` (USP).
- Colab / outra pessoa da equipe: `default` (abre o login da sua conta).

In [ ]:
CONTA = 'default' if EM_COLAB else 'conta1'
if CONTA == 'default':
    ee.Authenticate()
auth.init(CONTA)
print('conta:', CONTA, '| projeto:', cfg['cloud_project'], '| destino:', config.destination_root())

## 2 · Parâmetros
Carta piloto e ano. A região do mosaico é a **super carta** (a carta e suas vizinhas), igual à Col 11.

In [ ]:
CARTA = 'SF-23-Y-C'
ANO = 2020

contexto = grids.context(CARTA)
img = mosaico = mosaic.get(ANO, contexto)
print('fonte do mosaico:', cfg['mosaic']['source'])
print(len(mosaic.BANDS), 'bandas:', mosaic.BANDS)

## 3 · Visualização
Mapa interativo da super carta: composição RGB e alguns índices. Ligue/desligue camadas no canto do mapa.

In [ ]:
import geemap
m = geemap.Map()
m.centerObject(contexto, 9)
m.addLayer(mosaico, {'bands': ['RED', 'GREEN', 'BLUE'], 'min': 1000, 'max': 1290}, 'RGB')
m.addLayer(mosaico.select('NDBI'), {'min': 700, 'max': 1300, 'palette': ['white', 'red']}, 'NDBI', False)
m.addLayer(mosaico.select('NDVI'), {'min': 900, 'max': 1900, 'palette': ['white', 'green']}, 'NDVI', False)
m.addLayer(ee.FeatureCollection(cfg['grids']['cartas']).filter(ee.Filter.eq('grid_name', CARTA)).style(color='yellow', fillColor='00000000'), {}, CARTA)
m

## 4 · Conferência com a Col 11
Compara os valores deste mosaico com os do JS original num ponto.

1. Rode `verificacao/mosaico_col11_ponto.js` no Code Editor (mesma carta, ano e ponto).
2. Cole o dicionário impresso no Console em `VALORES_JS` abaixo.
3. Diferença zero em todas as bandas = porte fiel.

Aqui o mosaico é recalculado com `encoding='col11'` (a codificação antiga, com saturação), porque é isso que o JS produz. No resto do pipeline vale o `encoding` do YAML.

In [ ]:
PONTO = ee.Geometry.Point([-46.633, -23.55])
# a conferência usa a codificação idêntica à Col 11
mosaico_col11 = mosaic.on_the_fly(ANO, contexto, encoding='col11')
valores_py = mosaico_col11.reduceRegion(ee.Reducer.first(), PONTO, 30).getInfo()

VALORES_JS = {}   # cole aqui o dicionário do Code Editor

import pandas as pd
comp = pd.DataFrame({'python': valores_py, 'js_col11': pd.Series(VALORES_JS, dtype='float')}).reindex(mosaic.BANDS)
comp['diferenca'] = comp['python'] - comp['js_col11']
comp

## 5 · (Opcional) Exportar mosaico como asset
Grava o mosaico on-the-fly como asset para depois usar `mosaic.source: asset`.
**Só cria e inicia a tarefa quando `EXPORTAR = True`.** O destino segue `destination` do YAML (a pasta `mosaics/` precisa existir).

In [ ]:
EXPORTAR = False
if EXPORTAR:
    regiao = grids.region(CARTA)
    tarefa = mosaic.export_to_asset(ANO, regiao, f'mosaic_{ANO}_{CARTA}')
    tarefa.start()
    print('tarefa iniciada:', tarefa.status()['description'])
else:
    print('exportação desligada (EXPORTAR = False)')

## 6 · Acompanhar tarefas
Rode de novo sempre que quiser atualizar.

In [ ]:
tasks.status(CONTA).head(20)